In [ ]:
# 强制重新加载所有模块
import sys
modules_to_reload = ['config', 'util', 'data_preprocessing', 'sentiment_analysis', 'correlation_analysis', 'visualization', 'report_generator', 'data_ingestion']
for module in modules_to_reload:
    if module in sys.modules:
        del sys.modules[module]

# 重新导入所有模块
import config
from util import DataSaver
from data_preprocessing import DataPreprocessor
from sentiment_analysis import SentimentAnalyzer
from correlation_analysis import CorrelationEngine
from visualization import Visualizer
from report_generator import ReportGenerator

In [ ]:
import pandas as pd
import os

In [ ]:
from data_ingestion import PriceDataFetcher,CommentScraper
raw_price_df = PriceDataFetcher(token=config.TUSHARE_TOKEN).get_data(config.STOCK_CODE, config.START_DATE, config.END_DATE)
raw_comment_df = CommentScraper().get_data(config. STOCK_CODE, config.START_DATE, config.END_DATE)

In [ ]:
# 弃用
def load_raw_data(stock_code):
    """从CSV文件加载原始的价格和评论数据。"""
    print("--- 步骤 1: 加载原始数据 ---")
    price_path = os.path.join(config.RAW_DATA_PATH, f'{stock_code}_price_raw.csv')
    comment_path = os.path.join(config.RAW_DATA_PATH, f'{stock_code}_comment_raw.csv')

    if not os.path.exists(price_path) or not os.path.exists(comment_path):
        print(f"[错误] 原始数据文件不存在。请先运行 prepare_data.py 来生成数据。")
        print(f"  - 期望的价格文件: {price_path}")
        print(f"  - 期望的评论文件: {comment_path}")
        return None, None

    try:
        raw_price_df = pd.read_csv(price_path)
        raw_comment_df = pd.read_csv(comment_path)
        print("原始数据加载成功。")
        return raw_price_df, raw_comment_df
    except Exception as e:
        print(f"[错误] 加载原始数据文件时出错: {e}")
        return None, None

In [ ]:
def preprocess_data(raw_price_df, raw_comment_df, stock_code, save=False):
    """预处理价格和评论数据，并保存中间结果。"""
    print("--- 步骤 2.1: 数据预处理 ---")
    saver = DataSaver(save)

    # 预处理价格数据
    preprocessor = DataPreprocessor()
    processed_price_df = preprocessor.process_prices(raw_price_df.copy())
    saver.save_to_csv(processed_price_df.reset_index(), config.PROCESSED_DATA_PATH, f'{stock_code}_price_processed.csv')

    # 预处理评论数据
    processed_comment_df = preprocessor.process_comments(raw_comment_df.copy())
    saver.save_to_csv(processed_comment_df, config.PROCESSED_DATA_PATH, f'{stock_code}_comments_processed.csv')

    if processed_comment_df.empty:
        print("错误：评论数据预处理后为空。")
        return None, None

    print("数据预处理完成")
    if saver.status:
        print("中间结果已保存。")
    return processed_price_df, processed_comment_df

In [ ]:
def analyze_sentiment(processed_comment_df, stock_code, save=False):
    """对预处理后的评论数据进行增量情感分析，并保存结果。"""
    print("--- 步骤 2.2: 增量情感分析 ---")

    saver = DataSaver(save)

    # 创建情感分析器
    sentiment_analyzer = SentimentAnalyzer(model_path=config.SENTIMENT_MODEL_PATH)

    # 执行增量情感分析
    complete_sentiment_df, new_analyzed_count = sentiment_analyzer.analyze_with_incremental_processing(
        processed_comment_df, stock_code
    )

    # 保存完整的情感分析结果
    if saver.save_to_csv(complete_sentiment_df, config.PROCESSED_DATA_PATH,
                         f'{stock_code}_comments_with_sentiment.csv'):
        print(f"保存了 {len(complete_sentiment_df)} 条情感分析结果（新增 {new_analyzed_count} 条）")

    # 按日聚合情感数据
    daily_sentiment_df = sentiment_analyzer.aggregate_sentiment_daily(complete_sentiment_df)

    saver.save_to_csv(daily_sentiment_df.reset_index(), config.PROCESSED_DATA_PATH,
                         f'{stock_code}_daily_sentiment.csv')

    if daily_sentiment_df.empty:
        print("错误：没有可分析的情感数据。")
        return None

    print("增量情感分析完成")
    if saver.status:
        print("中间结果已保存。")

    return daily_sentiment_df,complete_sentiment_df

In [ ]:
def perform_correlation_analysis(processed_price_df, daily_sentiment_df, stock_code):
    """执行相关性分析，并保存中间结果。"""
    print("--- 步骤 3: 相关性分析 ---")
    saver = DataSaver()
    engine = CorrelationEngine()
    price_return_series = processed_price_df['pct_change']
    sentiment_series = daily_sentiment_df['daily_sentiment_score']

    aligned_return, aligned_sentiment = engine.align_series(price_return_series, sentiment_series)

    aligned_df = pd.DataFrame({'aligned_return': aligned_return, 'aligned_sentiment': aligned_sentiment})
    saver.save_to_csv(aligned_df.reset_index(), config.PROCESSED_DATA_PATH, f'{stock_code}_aligned_return_sentiment.csv')

    if aligned_return.empty or aligned_sentiment.empty:
        print("错误：价格和情感数据对齐后为空，无法进行相关性分析。")
        return None, None, None

    overall_corr = aligned_return.corr(aligned_sentiment)
    with open(os.path.join(config.PROCESSED_DATA_PATH, f'{stock_code}_overall_correlation.txt'), 'w') as f:
        f.write(str(overall_corr))
    print(f"Overall correlation saved.")

    rolling_corr = engine.calculate_rolling_correlation(aligned_return, aligned_sentiment, window=30).dropna()
    saver.save_to_csv(rolling_corr.to_frame(name='rolling_corr').reset_index(), config.PROCESSED_DATA_PATH, f'{stock_code}_rolling_correlation.csv')

    lagged_corr = engine.calculate_lagged_correlation(aligned_return, aligned_sentiment, max_lag=10)
    saver.save_to_csv(lagged_corr.to_frame(name='lagged_corr').reset_index(), config.PROCESSED_DATA_PATH, f'{stock_code}_lagged_correlation.csv')

    print("相关性分析完成，中间文件已保存。")
    return overall_corr, rolling_corr, lagged_corr

In [ ]:
def generate_visualizations(processed_price_df, daily_sentiment_df, corr_metrics):
    """生成所有可视化图表。"""
    print("--- 步骤 4.1: 生成可视化图表 ---")
    visualizer = Visualizer()
    engine = CorrelationEngine()  # For alignment

    price_close_series = processed_price_df['close']
    sentiment_series = daily_sentiment_df['daily_sentiment_score']
    aligned_price, aligned_sentiment_for_plot = engine.align_series(price_close_series, sentiment_series)
    aligned_return, aligned_sentiment_for_scatter = engine.align_series(processed_price_df['pct_change'],
                                                                        sentiment_series)

    figures = {
        'price_sentiment': visualizer.create_sentiment_vs_price_fig(aligned_price, aligned_sentiment_for_plot,
                                                                    config.STOCK_NAME),
        'scatterplot': visualizer.create_correlation_scatterplot_fig(aligned_sentiment_for_scatter, aligned_return,
                                                                     config.STOCK_NAME),
        'rolling_corr': visualizer.create_rolling_correlation_fig(corr_metrics['rolling'], config.STOCK_NAME,
                                                                  window=30),
        'lagged_corr': visualizer.create_lagged_correlation_fig(corr_metrics['lagged'], config.STOCK_NAME)
    }

    print("可视化图表生成完成。")
    return figures

In [ ]:
def generate_html_report(corr_metrics, figures):
    """生成HTML分析报告。"""
    print("--- 步骤 4.2: 生成HTML报告 ---")

    # 准备报告上下文
    overall_corr = corr_metrics['overall']
    lagged_corr = corr_metrics['lagged']
    lead_corr = lagged_corr[lagged_corr.index > 0]
    # lag_corr_only = lagged_corr[lag_corr.index < 0]

    report_context = {
        'title': f'{config.STOCK_NAME} 情感与价格相关性分析报告',
        'stock_name': config.STOCK_NAME,
        'stock_code': config.STOCK_CODE,
        'start_date': config.START_DATE,
        'end_date': config.END_DATE,
        'generated_at': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),
        'corr_metrics': {
            'overall': overall_corr,
            'max_lead_corr': lead_corr.max() if not lead_corr.empty and lead_corr.notna().any() else None,
            'max_lead_day': lead_corr.idxmax() if not lead_corr.empty and lead_corr.notna().any() else 'N/A',
            # 'max_lag_corr': lag_corr_only.max() if not lag_corr_only.empty and lag_corr_only.notna().any() else None,
            # 'max_lag_day': lag_corr_only.idxmax() if not lag_corr_only.empty and lag_corr_only.notna().any() else 'N/A',
        },
        'figures': figures
    }

    # 生成HTML报告
    report_generator = ReportGenerator(report_path=config.REPORT_PATH, template_str=HTML_TEMPLATE)
    report_generator.generate_report(config.HTML_REPORT_FILENAME, report_context)
    print("HTML报告生成完毕。")

In [ ]:
HTML_TEMPLATE = """
<!DOCTYPE html>
<html lang="zh">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>{{ title }}</title>
    <style>
        body { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, "Helvetica Neue", Arial, sans-serif; line-height: 1.6; color: #333; background-color: #f8f9fa; margin: 0; padding: 20px; }
        .container { max-width: 1200px; margin: auto; background: #fff; padding: 30px; border-radius: 8px; box-shadow: 0 2px 10px rgba(0,0,0,0.05); }
        h1, h2 { color: #0056b3; border-bottom: 2px solid #0056b3; padding-bottom: 10px; }
        .grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(500px, 1fr)); gap: 20px; }
        .card { padding: 20px; border-radius: 8px; box-shadow: 0 1px 5px rgba(0,0,0,0.1); }
        .metric { font-size: 1.5em; font-weight: bold; color: #007bff; }
        footer { text-align: center; margin-top: 30px; color: #6c757d; font-size: 0.9em; }
    </style>
</head>
<body>
    <div class="container">
        <h1>{{ title }}</h1>
        <p>本报告旨在分析 <b>{{ stock_name }} ({{ stock_code }})</b> 在 <b>{{ start_date }}</b> 到 <b>{{ end_date }}</b> 期间，公众评论情感与股票价格变动之间的相关性。</p>

        <h2>关键指标摘要</h2>
        <div class="card">
            <p>情感与收益率整体相关系数: <span class="metric">{{ '%.4f'|format(corr_metrics.overall) if corr_metrics.overall is not none else 'N/A' }}</span></p>
            <p>情感领先价格相关性峰值: <span class="metric">{{ '%.4f'|format(corr_metrics.max_lead_corr) if corr_metrics.max_lead_corr is not none else 'N/A' }}</span> (领先 {{ corr_metrics.max_lead_day }} 天)</p>
            <p>价格影响情绪相关性峰值: <span class="metric">{{ '%.4f'|format(corr_metrics.max_lag_corr) if corr_metrics.max_lag_corr is not none else 'N/A' }}</span> (滞后 {{ corr_metrics.max_lag_day }} 天)</p>
        </div>

        <h2>可视化分析</h2>
        <div class="grid">
            <div class="card">{{ figures.price_sentiment | safe }}</div>
            <div class="card">{{ figures.scatterplot | safe }}</div>
            <div class="card">{{ figures.rolling_corr | safe }}</div>
            <div class="card">{{ figures.lagged_corr | safe }}</div>
        </div>

        <footer>报告生成于 {{ generated_at }}</footer>
    </div>
</body>
</html>
"""
def generate_final_report(processed_price_df, daily_sentiment_df, corr_metrics):
    """生成可视化图表和最终的HTML报告。"""
    print("--- 步骤 4: 生成分析报告 ---")
    # 4.1 数据可视化
    visualizer = Visualizer()
    engine = CorrelationEngine()  # For alignment

    price_close_series = processed_price_df['close']
    sentiment_series = daily_sentiment_df['daily_sentiment_score']
    aligned_price, aligned_sentiment_for_plot = engine.align_series(price_close_series, sentiment_series)
    aligned_return, aligned_sentiment_for_scatter = engine.align_series(processed_price_df['pct_change'],
                                                                        sentiment_series)

    figures = {
        'price_sentiment': visualizer.create_sentiment_vs_price_fig(aligned_price, aligned_sentiment_for_plot,
                                                                    config.STOCK_NAME),
        'scatterplot': visualizer.create_correlation_scatterplot_fig(aligned_sentiment_for_scatter, aligned_return,
                                                                     config.STOCK_NAME),
        'rolling_corr': visualizer.create_rolling_correlation_fig(corr_metrics['rolling'], config.STOCK_NAME,
                                                                  window=30),
        'lagged_corr': visualizer.create_lagged_correlation_fig(corr_metrics['lagged'], config.STOCK_NAME)
    }

    # 4.2 准备报告上下文
    overall_corr = corr_metrics['overall']
    lagged_corr = corr_metrics['lagged']
    lead_corr = lagged_corr[lagged_corr.index > 0]
    lag_corr_only = lagged_corr[lag_corr.index < 0]

    report_context = {
        'title': f'{config.STOCK_NAME} 情感与价格相关性分析报告',
        'stock_name': config.STOCK_NAME,
        'stock_code': config.STOCK_CODE,
        'start_date': config.START_DATE,
        'end_date': config.END_DATE,
        'generated_at': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),
        'corr_metrics': {
            'overall': overall_corr,
            'max_lead_corr': lead_corr.max() if not lead_corr.empty and lead_corr.notna().any() else None,
            'max_lead_day': lead_corr.idxmax() if not lead_corr.empty and lead_corr.notna().any() else 'N/A',
            'max_lag_corr': lag_corr_only.max() if not lag_corr_only.empty and lag_corr_only.notna().any() else None,
            'max_lag_day': lag_corr_only.idxmax() if not lag_corr_only.empty and lag_corr_only.notna().any() else 'N/A',
        },
        'figures': figures
    }

    # 4.3 生成HTML报告
    report_generator = ReportGenerator(report_path=config.REPORT_PATH, template_str=HTML_TEMPLATE)
    report_generator.generate_report(config.HTML_REPORT_FILENAME, report_context)
    print("报告生成完毕。")

### ___MAIN___  **操作**

In [ ]:
raw_price_df, raw_comment_df = load_raw_data(stock_code=config.STOCK_CODE)

In [ ]:
processed_price_df, processed_comment_df = preprocess_data(raw_price_df, raw_comment_df, stock_code=config.STOCK_CODE, save=True)

In [ ]:
daily_sentiment_df, complete_sentiment_df = test_incremental_price_fetcher(processed_comment_df, stock_code=config.STOCK_CODE, save=True)

In [ ]:
daily_sentiment_df, complete_sentiment_df = daily_sentiment_df

In [ ]:
overall_corr, rolling_corr, lagged_corr = perform_correlation_analysis(processed_price_df, daily_sentiment_df, config.STOCK_CODE)

In [ ]:
# 数据可视化
# 相关性数据打包成字典,方便传参. 后面可能会遇到更多复杂的分析数据,或许都需要打包吧
correlation_metrics = {
    'overall': overall_corr,
    'rolling': rolling_corr,
    'lagged': lagged_corr
}
# 这个函数可以拆开
figures = generate_visualizations(processed_price_df, daily_sentiment_df, correlation_metrics)

In [ ]:
generate_html_report(correlation_metrics, figures)

## MODEL

In [ ]:
daily_sentiment_df

In [ ]:
processed_price_df

In [ ]:
# 在你现有的情感分析之后添加
from advanced_sentiment_analysis import run_advanced_sentiment_analysis
from temporal_sentiment_analysis import run_comprehensive_sentiment_analysis

# 运行高级分析 - 修复：使用包含情感分析结果的DataFrame
advanced_results = run_advanced_sentiment_analysis(
    complete_sentiment_df, daily_sentiment_df, processed_price_df
)

# 运行时间序列分析 - 修复：同样使用包含情感分析结果的DataFrame
temporal_results = run_comprehensive_sentiment_analysis(
    complete_sentiment_df, daily_sentiment_df
)

In [ ]:
# 强制清理所有缓存并重新加载
import sys
import importlib

# 删除所有相关模块的缓存
modules_to_clear = [
    'temporal_sentiment_analysis',
    'advanced_visualization',
    'advanced_sentiment_analysis'
]

for module in modules_to_clear:
    if module in sys.modules:
        del sys.modules[module]
        print(f"已清理模块: {module}")

# 强制垃圾回收
import gc
gc.collect()

# 重新导入模块
from temporal_sentiment_analysis import run_comprehensive_sentiment_analysis
from advanced_visualization import run_advanced_visualization
from advanced_sentiment_analysis import run_advanced_sentiment_analysis

print("✅ 所有模块已重新加载")

In [ ]:
# 强制清理并重新加载修复后的模块
import sys
import gc

modules_to_clear = ['temporal_sentiment_analysis', 'advanced_visualization']
for module in modules_to_clear:
    if module in sys.modules:
        del sys.modules[module]

gc.collect()

# 重新导入修复后的模块
from temporal_sentiment_analysis import run_comprehensive_sentiment_analysis
from advanced_visualization import run_advanced_visualization

print("✅ 修复后的模块已重新加载")

In [ ]:
# 重新运行时间序列分析（现在包含完整的动量指标）
print("🔄 重新运行时间序列分析...")
temporal_results = run_comprehensive_sentiment_analysis(complete_sentiment_df, daily_sentiment_df)

print("🎨 开始生成可视化图表...")
visualization_figures = run_advanced_visualization(advanced_results, temporal_results)

print("🎉 高级情感分析可视化完成！")
print(f"✅ 共生成了 {len(visualization_figures)} 个可视化图表")
print("📂 所有图表已保存到 reports/figures/ 目录")

In [ ]:
# 在notebook中重新运行分析
import sys
sys.modules.pop('advanced_sentiment_analysis', None)
from advanced_sentiment_analysis import run_advanced_sentiment_analysis

# 重新运行高级分析
advanced_results = run_advanced_sentiment_analysis(
    complete_sentiment_df, daily_sentiment_df, processed_price_df
)

In [ ]:
print("🔄 重新运行时间序列分析...")
temporal_results = run_comprehensive_sentiment_analysis(complete_sentiment_df, daily_sentiment_df)

print("🎨 开始生成可视化图表...")
visualization_figures = run_advanced_visualization(advanced_results, temporal_results)

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import os
import config

import pandas as pd
import numpy as np
from datetime import datetime
import os
import config

class FeatureEngineer:
    """
    特征工程器 - 整合所有分析结果
    """

    def __init__(self):
        print("特征工程器初始化完成")

    def create_comprehensive_daily_features(self,
                                          price_df,
                                          daily_sentiment_df,
                                          advanced_results,
                                          temporal_results):
        """
        创建综合的每日特征表格

        参数:
        - price_df: 股价数据
        - daily_sentiment_df: 每日情感数据
        - advanced_results: 高级情感分析结果
        - temporal_results: 时间序列分析结果

        返回:
        - comprehensive_df: 综合特征DataFrame
        """
        print("开始创建综合每日特征表格...")

        # 1. 基础价格特征
        price_features = self._extract_price_features(price_df)
        print(f"✅ 提取价格特征: {len(price_features.columns)} 列")

        # 2. 基础情感特征
        sentiment_features = self._extract_sentiment_features(daily_sentiment_df)
        print(f"✅ 提取情感特征: {len(sentiment_features.columns)} 列")

        # 3. 高级情感特征
        advanced_features = self._extract_advanced_features(advanced_results)
        print(f"✅ 提取高级特征: {len(advanced_features.columns)} 列")

        # 4. 时间序列特征
        temporal_features = self._extract_temporal_features(temporal_results)
        print(f"✅ 提取时间特征: {len(temporal_features.columns)} 列")

        # 5. 技术指标特征
        technical_features = self._extract_technical_features(price_df)
        print(f"✅ 提取技术特征: {len(technical_features.columns)} 列")

        # 6. 合并所有特征
        comprehensive_df = self._merge_all_features([
            price_features,
            sentiment_features,
            advanced_features,
            temporal_features,
            technical_features
        ])

        # 7. 添加时间特征
        comprehensive_df = self._add_time_features(comprehensive_df)

        # 8. 计算衍生特征
        comprehensive_df = self._calculate_derived_features(comprehensive_df)

        print(f"🎉 综合特征表格创建完成！总共 {len(comprehensive_df.columns)} 列，{len(comprehensive_df)} 行")

        return comprehensive_df

    def _extract_price_features(self, price_df):
        """提取价格相关特征 - 适应不同的列名格式"""
        df = price_df.copy()

        # 确保日期为索引
        if 'date' in df.columns:
            df = df.set_index('date')

        price_features = pd.DataFrame(index=df.index)

        # 基础价格数据 - 处理可能的列名变化
        price_features['开盘价'] = df['open']
        price_features['最高价'] = df['high']
        price_features['最低价'] = df['low']
        price_features['收盘价'] = df['close']

        # 成交量 - 尝试不同的列名
        volume_col = None
        for col_name in ['volume', 'vol', '成交量', 'turnover']:
            if col_name in df.columns:
                volume_col = col_name
                break

        if volume_col:
            price_features['成交量'] = df[volume_col]
            price_features['成交额'] = df.get('amount', df[volume_col] * df['close'])
        else:
            print("⚠️ 警告：未找到成交量相关列，将使用默认值")
            price_features['成交量'] = 0
            price_features['成交额'] = 0

        # 价格变化
        price_features['日收益率'] = df['pct_change']
        price_features['价格变化额'] = df['close'].diff()
        price_features['振幅'] = (df['high'] - df['low']) / df['close'].shift(1)

        # 成交量特征 - 只有在有成交量数据时才计算
        if volume_col:
            price_features['成交量变化率'] = df[volume_col].pct_change()
            price_features['量价比'] = df[volume_col] / df['close']
        else:
            price_features['成交量变化率'] = 0
            price_features['量价比'] = 0

        return price_features

    def _extract_sentiment_features(self, daily_sentiment_df):
        """提取基础情感特征"""
        df = daily_sentiment_df.copy()

        # 确保日期为索引
        if not isinstance(df.index, pd.DatetimeIndex):
            df.index = pd.to_datetime(df.index)

        sentiment_features = pd.DataFrame(index=df.index)

        # 基础情感指标
        sentiment_features['每日情感分数'] = df['daily_sentiment_score']
        sentiment_features['评论数量'] = df['daily_comment_count']  # 修正字段名
        sentiment_features['正面评论数'] = df.get('positive_count', 0)
        sentiment_features['负面评论数'] = df.get('negative_count', 0)
        sentiment_features['中性评论数'] = df.get('neutral_count', 0)

        # 情感强度
        sentiment_features['情感强度'] = df['daily_sentiment_score'].abs()
        sentiment_features['情感方向'] = np.sign(df['daily_sentiment_score'])

        # 情感变化 - 添加前缀避免冲突
        sentiment_features['基础_情感变化'] = df['daily_sentiment_score'].diff()
        sentiment_features['基础_情感变化率'] = df['daily_sentiment_score'].pct_change()

        return sentiment_features

    def _extract_advanced_features(self, advanced_results):
        """提取高级情感分析特征"""
        features_list = []

        # 1. 主题情感特征
        if 'topic_sentiment' in advanced_results:
            topic_features = self._process_topic_sentiment(advanced_results['topic_sentiment'])
            features_list.append(topic_features)

        # 2. 情感强度特征
        if 'intensity_stats' in advanced_results:
            # 这个通常是统计数据，我们需要从原始数据重新计算每日强度
            pass

        # 3. 极端情感事件
        if 'extreme_events' in advanced_results:
            extreme_features = self._process_extreme_events(advanced_results['extreme_events'])
            features_list.append(extreme_features)

        # 4. 背离分析特征
        if 'divergence_df' in advanced_results:
            divergence_features = self._process_divergence_features(advanced_results['divergence_df'])
            features_list.append(divergence_features)

        # 合并所有高级特征
        if features_list:
            # 找到共同的日期索引
            common_index = features_list[0].index
            for df in features_list[1:]:
                common_index = common_index.intersection(df.index)

            advanced_features = pd.DataFrame(index=common_index)
            for df in features_list:
                advanced_features = advanced_features.join(df, how='left')
        else:
            # 创建空的DataFrame
            advanced_features = pd.DataFrame()

        return advanced_features

    def _process_topic_sentiment(self, topic_sentiment_dict):
        """处理主题情感数据"""
        topic_features = pd.DataFrame()

        for topic, sentiment_series in topic_sentiment_dict.items():
            col_name = f'{topic}_情感'
            topic_features[col_name] = sentiment_series

        # 计算主题情感的统计特征
        if len(topic_features.columns) > 0:
            topic_features['主题情感_均值'] = topic_features.mean(axis=1)
            topic_features['主题情感_标准差'] = topic_features.std(axis=1)
            topic_features['主题情感_最大值'] = topic_features.max(axis=1)
            topic_features['主题情感_最小值'] = topic_features.min(axis=1)

        return topic_features

    def _process_extreme_events(self, extreme_events_df):
        """处理极端情感事件"""
        extreme_features = pd.DataFrame(index=extreme_events_df.index)

        extreme_features['极端正面事件数'] = extreme_events_df['extreme_positive_count']
        extreme_features['极端负面事件数'] = extreme_events_df['extreme_negative_count']
        extreme_features['极端事件总数'] = (extreme_events_df['extreme_positive_count'] +
                                    extreme_events_df['extreme_negative_count'])
        extreme_features['极端事件净值'] = (extreme_events_df['extreme_positive_count'] -
                                    extreme_events_df['extreme_negative_count'])

        return extreme_features

    def _process_divergence_features(self, divergence_df):
        """处理背离分析特征"""
        divergence_features = pd.DataFrame(index=divergence_df.index)

        divergence_features['情感_标准化'] = divergence_df['sentiment']
        divergence_features['收益率_标准化'] = divergence_df['returns']
        divergence_features['情感方向'] = divergence_df['sentiment_direction']
        divergence_features['价格方向'] = divergence_df['price_direction']
        divergence_features['是否背离'] = divergence_df['is_divergence'].astype(int)

        return divergence_features

    def _extract_temporal_features(self, temporal_results):
        """提取时间序列特征"""
        features_list = []

        # 1. 情感动量特征
        if 'momentum_analysis' in temporal_results:
            momentum_features = self._process_momentum_features(temporal_results['momentum_analysis'])
            features_list.append(momentum_features)
        elif 'momentum_signals' in temporal_results:
            momentum_features = self._process_momentum_features(temporal_results['momentum_signals'])
            features_list.append(momentum_features)

        # 2. 反转信号特征
        if 'reversal_signals' in temporal_results:
            reversal_features = self._process_reversal_features(temporal_results['reversal_signals'])
            features_list.append(reversal_features)

        # 合并时间特征
        if features_list:
            common_index = features_list[0].index
            for df in features_list[1:]:
                common_index = common_index.intersection(df.index)

            temporal_features = pd.DataFrame(index=common_index)
            for df in features_list:
                temporal_features = temporal_features.join(df, how='left')
        else:
            temporal_features = pd.DataFrame()

        return temporal_features

    def _process_momentum_features(self, momentum_df):
        """处理动量特征"""
        momentum_features = pd.DataFrame(index=momentum_df.index)

        momentum_features['情感移动平均'] = momentum_df['sentiment_ma']
        momentum_features['情感动量'] = momentum_df['sentiment_momentum']
        momentum_features['动量_情感变化率'] = momentum_df['sentiment_change_rate']  # 添加前缀避免冲突
        momentum_features['情感波动率'] = momentum_df['sentiment_volatility']

        # 如果有交易信号
        if 'sentiment_signal' in momentum_df.columns:
            momentum_features['交易信号'] = momentum_df['sentiment_signal']
            # 将交易信号转换为数值
            signal_map = {'强烈看多': 2, '温和看多': 1, '中性': 0, '温和看空': -1, '强烈看空': -2}
            momentum_features['交易信号_数值'] = momentum_df['sentiment_signal'].map(signal_map).fillna(0)

        return momentum_features

    def _process_reversal_features(self, reversal_df):
        """处理反转特征"""
        reversal_features = pd.DataFrame(index=reversal_df.index)

        reversal_features['是否反转'] = reversal_df['is_reversal'].astype(int)

        if 'reversal_type' in reversal_df.columns:
            reversal_features['反转类型'] = reversal_df['reversal_type']
            # 转换为数值
            reversal_map = {'向上反转': 1, '向下反转': -1, '无反转': 0}
            reversal_features['反转类型_数值'] = reversal_df['reversal_type'].map(reversal_map).fillna(0)

        return reversal_features

    def _extract_technical_features(self, price_df):
        """提取技术指标特征 - 适应不同的列名格式"""
        df = price_df.copy()

        if 'date' in df.columns:
            df = df.set_index('date')

        technical_features = pd.DataFrame(index=df.index)

        # 移动平均线
        technical_features['MA5'] = df['close'].rolling(5).mean()
        technical_features['MA10'] = df['close'].rolling(10).mean()
        technical_features['MA20'] = df['close'].rolling(20).mean()

        # 价格相对位置
        technical_features['价格_MA5_比'] = df['close'] / technical_features['MA5']
        technical_features['价格_MA20_比'] = df['close'] / technical_features['MA20']

        # 成交量移动平均 - 尝试不同的列名
        volume_col = None
        for col_name in ['volume', 'vol', '成交量', 'turnover']:
            if col_name in df.columns:
                volume_col = col_name
                break

        if volume_col:
            technical_features['成交量_MA5'] = df[volume_col].rolling(5).mean()
            technical_features['成交量_相对强度'] = df[volume_col] / technical_features['成交量_MA5']
        else:
            technical_features['成交量_MA5'] = 0
            technical_features['成交量_相对强度'] = 0

        # RSI (简化版)
        delta = df['close'].diff()
        gain = (delta.where(delta > 0, 0)).rolling(14).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
        rs = gain / loss
        technical_features['RSI'] = 100 - (100 / (1 + rs))

        # 布林带
        rolling_mean = df['close'].rolling(20).mean()
        rolling_std = df['close'].rolling(20).std()
        technical_features['布林上轨'] = rolling_mean + (rolling_std * 2)
        technical_features['布林下轨'] = rolling_mean - (rolling_std * 2)
        technical_features['布林位置'] = (df['close'] - technical_features['布林下轨']) / (technical_features['布林上轨'] - technical_features['布林下轨'])

        return technical_features

    def _merge_all_features(self, feature_dfs):
        """合并所有特征DataFrame"""
        # 过滤掉空的DataFrame
        valid_dfs = [df for df in feature_dfs if not df.empty]

        if not valid_dfs:
            return pd.DataFrame()

        # 找到共同的日期范围
        common_index = valid_dfs[0].index
        for df in valid_dfs[1:]:
            common_index = common_index.intersection(df.index)

        # 基于共同日期范围合并
        comprehensive_df = pd.DataFrame(index=common_index)

        for df in valid_dfs:
            comprehensive_df = comprehensive_df.join(df, how='left')

        return comprehensive_df

    def _add_time_features(self, df):
        """添加时间特征"""
        if df.empty:
            return df

        df = df.copy()

        # 时间特征
        df['年'] = df.index.year
        df['月'] = df.index.month
        df['日'] = df.index.day
        df['星期'] = df.index.dayofweek  # 0=Monday, 6=Sunday
        df['是否周末'] = (df.index.dayofweek >= 5).astype(int)
        df['季度'] = df.index.quarter
        df['年内第几天'] = df.index.dayofyear

        return df

    def _calculate_derived_features(self, df):
        """计算衍生特征"""
        if df.empty:
            return df

        df = df.copy()

        # 1. 价格动量特征
        if '收盘价' in df.columns:
            df['价格动量_3日'] = df['收盘价'].rolling(3).apply(lambda x: x.iloc[-1] - x.iloc[0] if len(x) == 3 else np.nan)
            df['价格动量_5日'] = df['收盘价'].rolling(5).apply(lambda x: x.iloc[-1] - x.iloc[0] if len(x) == 5 else np.nan)

        # 2. 情感-价格相关性特征
        if '每日情感分数' in df.columns and '日收益率' in df.columns:
            # 滚动相关性
            window = 10
            df['情感价格相关性_10日'] = df['每日情感分数'].rolling(window).corr(df['日收益率'])

            # 情感与收益率的乘积（同向性指标）
            df['情感收益同向性'] = df['每日情感分数'] * df['日收益率']

        # 3. 综合信号特征
        if '交易信号_数值' in df.columns and 'RSI' in df.columns:
            df['综合信号'] = (df['交易信号_数值'] + (df['RSI'] - 50) / 25) / 2  # 标准化RSI并结合

        # 4. 风险特征
        if '情感波动率' in df.columns and '振幅' in df.columns:
            df['综合波动率'] = (df['情感波动率'].fillna(0) + df['振幅'].fillna(0)) / 2

        return df

    def save_features(self, comprehensive_df, filename=None):
        """保存特征表格"""
        if filename is None:
            filename = f"{config.STOCK_CODE}_comprehensive_features.csv"

        filepath = os.path.join('data/processed', filename)
        comprehensive_df.to_csv(filepath, encoding='utf-8-sig')
        print(f"✅ 特征表格已保存到: {filepath}")

        return filepath

    def print_feature_summary(self, comprehensive_df):
        """打印特征总结"""
        print("\n" + "="*80)
        print("📊 综合特征表格总结")
        print("="*80)

        print(f"📅 时间范围: {comprehensive_df.index.min().strftime('%Y-%m-%d')} 至 {comprehensive_df.index.max().strftime('%Y-%m-%d')}")
        print(f"📊 数据行数: {len(comprehensive_df)} 天")
        print(f"📈 特征列数: {len(comprehensive_df.columns)} 个")

        print(f"\n🔍 特征分类统计:")

        # 按特征类型分类
        feature_categories = {
            '价格特征': ['开盘价', '最高价', '最低价', '收盘价', '成交量', '成交额', '日收益率', '价格变化额', '振幅', '成交量变化率', '量价比'],
            '情感特征': ['每日情感分数', '评论数量', '正面评论数', '负面评论数', '中性评论数', '情感强度', '情感方向', '情感变化', '情感变化率'],
            '主题特征': [col for col in comprehensive_df.columns if '_情感' in col or '主题情感' in col],
            '极端事件': [col for col in comprehensive_df.columns if '极端' in col],
            '技术指标': ['MA5', 'MA10', 'MA20', 'RSI', '布林上轨', '布林下轨', '布林位置'] + [col for col in comprehensive_df.columns if 'MA' in col or '价格_' in col],
            '动量特征': [col for col in comprehensive_df.columns if '动量' in col or '波动' in col],
            '时间特征': ['年', '月', '日', '星期', '是否周末', '季度', '年内第几天'],
            '衍生特征': [col for col in comprehensive_df.columns if '相关性' in col or '同向性' in col or '综合' in col],
            '交易信号': [col for col in comprehensive_df.columns if '信号' in col or '反转' in col or '背离' in col]
        }

        for category, features in feature_categories.items():
            existing_features = [f for f in features if f in comprehensive_df.columns]
            if existing_features:
                print(f"  {category}: {len(existing_features)} 个")
                print(f"    {', '.join(existing_features[:3])}{'...' if len(existing_features) > 3 else ''}")

        print(f"\n📋 数据质量:")
        missing_rate = comprehensive_df.isnull().sum().sum() / (len(comprehensive_df) * len(comprehensive_df.columns))
        print(f"  缺失值比例: {missing_rate:.2%}")
        print(f"  完整数据行数: {comprehensive_df.dropna().shape[0]} 行")

        print("\n" + "="*80)


def create_comprehensive_features(price_df, daily_sentiment_df, advanced_results, temporal_results):
    """
    创建综合特征表格的主函数

    返回:
    - comprehensive_df: 综合特征DataFrame
    """
    engineer = FeatureEngineer()

    # 创建综合特征
    comprehensive_df = engineer.create_comprehensive_daily_features(
        price_df, daily_sentiment_df, advanced_results, temporal_results
    )

    # 打印特征总结
    engineer.print_feature_summary(comprehensive_df)

    # 保存特征表格
    filepath = engineer.save_features(comprehensive_df)

    return comprehensive_df, filepath


In [ ]:
# 强制清理所有缓存并重新加载
import sys
import importlib

# 删除所有相关模块的缓存
modules_to_clear = [
    'temporal_sentiment_analysis',
    'advanced_visualization',
    'advanced_sentiment_analysis',
    'feature_engineering'
]

for module in modules_to_clear:
    if module in sys.modules:
        del sys.modules[module]
        print(f"已清理模块: {module}")

# 强制垃圾回收
import gc
gc.collect()

# 重新导入模块
from temporal_sentiment_analysis import run_comprehensive_sentiment_analysis
from advanced_visualization import run_advanced_visualization
from advanced_sentiment_analysis import run_advanced_sentiment_analysis

print("✅ 所有模块已重新加载")

In [ ]:
# 强制清理并重新加载修复后的模块
import sys
import gc

modules_to_clear = ['temporal_sentiment_analysis', 'advanced_visualization', 'feature_engineering']
for module in modules_to_clear:
    if module in sys.modules:
        del sys.modules[module]

gc.collect()

# 重新导入修复后的模块
from temporal_sentiment_analysis import run_comprehensive_sentiment_analysis
from advanced_visualization import run_advanced_visualization
from feature_engineering import create_comprehensive_features

print("✅ 修复后的模块已重新加载")

In [ ]:
# 重新运行时间序列分析（现在包含完整的动量指标）
print("🔄 重新运行时间序列分析...")
temporal_results = run_comprehensive_sentiment_analysis(complete_sentiment_df, daily_sentiment_df)

print("🎨 开始生成可视化图表...")
visualization_figures = run_advanced_visualization(advanced_results, temporal_results)

print("🎉 高级情感分析可视化完成！")
print(f"✅ 共生成了 {len(visualization_figures)} 个可视化图表")
print("📂 所有图表已保存到 reports/figures/ 目录")

In [ ]:
# 在notebook中重新运行分析
import sys
sys.modules.pop('advanced_sentiment_analysis', None)
from advanced_sentiment_analysis import run_advanced_sentiment_analysis

# 重新运行高级分析
advanced_results = run_advanced_sentiment_analysis(
    complete_sentiment_df, daily_sentiment_df, processed_price_df
)

In [ ]:
print("🔄 重新运行时间序列分析...")
temporal_results = run_comprehensive_sentiment_analysis(complete_sentiment_df, daily_sentiment_df)

print("🎨 开始生成可视化图表...")
visualization_figures = run_advanced_visualization(advanced_results, temporal_results)

In [ ]:
# 先检查数据结构
print("🔍 检查数据结构...")
print(f"processed_price_df 列名: {list(processed_price_df.columns)}")
print(f"processed_price_df 索引: {processed_price_df.index.name}")
print(f"processed_price_df 形状: {processed_price_df.shape}")
print("\nprocessed_price_df 前几行:")
print(processed_price_df.head())
print("\n" + "="*50)

print(f"daily_sentiment_df 列名: {list(daily_sentiment_df.columns)}")
print(f"daily_sentiment_df 索引: {daily_sentiment_df.index.name}")
print(f"daily_sentiment_df 形状: {daily_sentiment_df.shape}")
print("\ndaily_sentiment_df 前几行:")
print(daily_sentiment_df.head())


In [ ]:
# -*- coding: utf-8 -*-
"""
特征工程模块
整合所有分析结果为按日的特征表格
"""

import pandas as pd
import numpy as np
from datetime import datetime
import os
import config

class FeatureEngineer:
    """
    特征工程器 - 整合所有分析结果
    """

    def __init__(self):
        print("特征工程器初始化完成")

    def create_comprehensive_daily_features(self,
                                          price_df,
                                          daily_sentiment_df,
                                          advanced_results,
                                          temporal_results):
        """
        创建综合的每日特征表格

        参数:
        - price_df: 股价数据
        - daily_sentiment_df: 每日情感数据
        - advanced_results: 高级情感分析结果
        - temporal_results: 时间序列分析结果

        返回:
        - comprehensive_df: 综合特征DataFrame
        """
        print("开始创建综合每日特征表格...")

        # 1. 基础价格特征
        price_features = self._extract_price_features(price_df)
        print(f"✅ 提取价格特征: {len(price_features.columns)} 列")

        # 2. 基础情感特征
        sentiment_features = self._extract_sentiment_features(daily_sentiment_df)
        print(f"✅ 提取情感特征: {len(sentiment_features.columns)} 列")

        # 3. 高级情感特征
        advanced_features = self._extract_advanced_features(advanced_results)
        print(f"✅ 提取高级特征: {len(advanced_features.columns)} 列")

        # 4. 时间序列特征
        temporal_features = self._extract_temporal_features(temporal_results)
        print(f"✅ 提取时间特征: {len(temporal_features.columns)} 列")

        # 5. 技术指标特征
        technical_features = self._extract_technical_features(price_df)
        print(f"✅ 提取技术特征: {len(technical_features.columns)} 列")

        # 6. 合并所有特征
        comprehensive_df = self._merge_all_features([
            price_features,
            sentiment_features,
            advanced_features,
            temporal_features,
            technical_features
        ])

        # 7. 添加时间特征
        comprehensive_df = self._add_time_features(comprehensive_df)

        # 8. 计算衍生特征
        comprehensive_df = self._calculate_derived_features(comprehensive_df)

        print(f"🎉 综合特征表格创建完成！总共 {len(comprehensive_df.columns)} 列，{len(comprehensive_df)} 行")

        return comprehensive_df

    def _extract_price_features(self, price_df):
        """提取价格相关特征 - 适应不同的列名格式"""
        df = price_df.copy()

        # 确保日期为索引
        if 'date' in df.columns:
            df = df.set_index('date')

        price_features = pd.DataFrame(index=df.index)

        # 基础价格数据 - 处理可能的列名变化
        price_features['开盘价'] = df['open']
        price_features['最高价'] = df['high']
        price_features['最低价'] = df['low']
        price_features['收盘价'] = df['close']

        # 成交量 - 尝试不同的列名
        volume_col = None
        for col_name in ['volume', 'vol', '成交量', 'turnover']:
            if col_name in df.columns:
                volume_col = col_name
                break

        if volume_col:
            price_features['成交量'] = df[volume_col]
            price_features['成交额'] = df.get('amount', df[volume_col] * df['close'])
        else:
            print("⚠️ 警告：未找到成交量相关列，将使用默认值")
            price_features['成交量'] = 0
            price_features['成交额'] = 0

        # 价格变化
        price_features['日收益率'] = df['pct_change']
        price_features['价格变化额'] = df['close'].diff()
        price_features['振幅'] = (df['high'] - df['low']) / df['close'].shift(1)

        # 成交量特征 - 只有在有成交量数据时才计算
        if volume_col:
            price_features['成交量变化率'] = df[volume_col].pct_change()
            price_features['量价比'] = df[volume_col] / df['close']
        else:
            price_features['成交量变化率'] = 0
            price_features['量价比'] = 0

        return price_features

    def _extract_sentiment_features(self, daily_sentiment_df):
        """提取基础情感特征"""
        df = daily_sentiment_df.copy()

        # 确保日期为索引
        if not isinstance(df.index, pd.DatetimeIndex):
            df.index = pd.to_datetime(df.index)

        sentiment_features = pd.DataFrame(index=df.index)

        # 基础情感指标
        sentiment_features['每日情感分数'] = df['daily_sentiment_score']
        sentiment_features['评论数量'] = df['daily_comment_count']  # 修正字段名
        sentiment_features['正面评论数'] = df.get('positive_count', 0)
        sentiment_features['负面评论数'] = df.get('negative_count', 0)
        sentiment_features['中性评论数'] = df.get('neutral_count', 0)

        # 情感强度
        sentiment_features['情感强度'] = df['daily_sentiment_score'].abs()
        sentiment_features['情感方向'] = np.sign(df['daily_sentiment_score'])

        # 情感变化 - 添加前缀避免冲突
        sentiment_features['基础_情感变化'] = df['daily_sentiment_score'].diff()
        sentiment_features['基础_情感变化率'] = df['daily_sentiment_score'].pct_change()

        return sentiment_features

    def _extract_advanced_features(self, advanced_results):
        """提取高级情感分析特征"""
        features_list = []

        # 1. 主题情感特征
        if 'topic_sentiment' in advanced_results:
            topic_features = self._process_topic_sentiment(advanced_results['topic_sentiment'])
            features_list.append(topic_features)

        # 2. 情感强度特征
        if 'intensity_stats' in advanced_results:
            # 这个通常是统计数据，我们需要从原始数据重新计算每日强度
            pass

        # 3. 极端情感事件
        if 'extreme_events' in advanced_results:
            extreme_features = self._process_extreme_events(advanced_results['extreme_events'])
            features_list.append(extreme_features)

        # 4. 背离分析特征
        if 'divergence_df' in advanced_results:
            divergence_features = self._process_divergence_features(advanced_results['divergence_df'])
            features_list.append(divergence_features)

        # 合并所有高级特征
        if features_list:
            # 找到共同的日期索引
            common_index = features_list[0].index
            for df in features_list[1:]:
                common_index = common_index.intersection(df.index)

            advanced_features = pd.DataFrame(index=common_index)
            for df in features_list:
                advanced_features = advanced_features.join(df, how='left')
        else:
            # 创建空的DataFrame
            advanced_features = pd.DataFrame()

        return advanced_features

    def _process_topic_sentiment(self, topic_sentiment_dict):
        """处理主题情感数据"""
        topic_features = pd.DataFrame()

        for topic, sentiment_series in topic_sentiment_dict.items():
            col_name = f'{topic}_情感'
            topic_features[col_name] = sentiment_series

        # 计算主题情感的统计特征
        if len(topic_features.columns) > 0:
            topic_features['主题情感_均值'] = topic_features.mean(axis=1)
            topic_features['主题情感_标准差'] = topic_features.std(axis=1)
            topic_features['主题情感_最大值'] = topic_features.max(axis=1)
            topic_features['主题情感_最小值'] = topic_features.min(axis=1)

        return topic_features

    def _process_extreme_events(self, extreme_events_df):
        """处理极端情感事件"""
        extreme_features = pd.DataFrame(index=extreme_events_df.index)

        extreme_features['极端正面事件数'] = extreme_events_df['extreme_positive_count']
        extreme_features['极端负面事件数'] = extreme_events_df['extreme_negative_count']
        extreme_features['极端事件总数'] = (extreme_events_df['extreme_positive_count'] +
                                    extreme_events_df['extreme_negative_count'])
        extreme_features['极端事件净值'] = (extreme_events_df['extreme_positive_count'] -
                                    extreme_events_df['extreme_negative_count'])

        return extreme_features

    def _process_divergence_features(self, divergence_df):
        """处理背离分析特征"""
        divergence_features = pd.DataFrame(index=divergence_df.index)

        divergence_features['情感_标准化'] = divergence_df['sentiment']
        divergence_features['收益率_标准化'] = divergence_df['returns']
        divergence_features['情感方向'] = divergence_df['sentiment_direction']
        divergence_features['价格方向'] = divergence_df['price_direction']
        divergence_features['是否背离'] = divergence_df['is_divergence'].astype(int)

        return divergence_features

    def _extract_temporal_features(self, temporal_results):
        """提取时间序列特征"""
        features_list = []

        # 1. 情感动量特征
        if 'momentum_analysis' in temporal_results:
            momentum_features = self._process_momentum_features(temporal_results['momentum_analysis'])
            features_list.append(momentum_features)
        elif 'momentum_signals' in temporal_results:
            momentum_features = self._process_momentum_features(temporal_results['momentum_signals'])
            features_list.append(momentum_features)

        # 2. 反转信号特征
        if 'reversal_signals' in temporal_results:
            reversal_features = self._process_reversal_features(temporal_results['reversal_signals'])
            features_list.append(reversal_features)

        # 合并时间特征
        if features_list:
            common_index = features_list[0].index
            for df in features_list[1:]:
                common_index = common_index.intersection(df.index)

            temporal_features = pd.DataFrame(index=common_index)
            for df in features_list:
                temporal_features = temporal_features.join(df, how='left')
        else:
            temporal_features = pd.DataFrame()

        return temporal_features

    def _process_momentum_features(self, momentum_df):
        """处理动量特征"""
        momentum_features = pd.DataFrame(index=momentum_df.index)

        momentum_features['情感移动平均'] = momentum_df['sentiment_ma']
        momentum_features['情感动量'] = momentum_df['sentiment_momentum']
        momentum_features['动量_情感变化率'] = momentum_df['sentiment_change_rate']  # 添加前缀避免冲突
        momentum_features['情感波动率'] = momentum_df['sentiment_volatility']

        # 如果有交易信号
        if 'sentiment_signal' in momentum_df.columns:
            momentum_features['交易信号'] = momentum_df['sentiment_signal']
            # 将交易信号转换为数值
            signal_map = {'强烈看多': 2, '温和看多': 1, '中性': 0, '温和看空': -1, '强烈看空': -2}
            momentum_features['交易信号_数值'] = momentum_df['sentiment_signal'].map(signal_map).fillna(0)

        return momentum_features

    def _process_reversal_features(self, reversal_df):
        """处理反转特征"""
        reversal_features = pd.DataFrame(index=reversal_df.index)

        reversal_features['是否反转'] = reversal_df['is_reversal'].astype(int)

        if 'reversal_type' in reversal_df.columns:
            reversal_features['反转类型'] = reversal_df['reversal_type']
            # 转换为数值
            reversal_map = {'向上反转': 1, '向下反转': -1, '无反转': 0}
            reversal_features['反转类型_数值'] = reversal_df['reversal_type'].map(reversal_map).fillna(0)

        return reversal_features

    def _extract_technical_features(self, price_df):
        """提取技术指标特征 - 适应不同的列名格式"""
        df = price_df.copy()

        if 'date' in df.columns:
            df = df.set_index('date')

        technical_features = pd.DataFrame(index=df.index)

        # 移动平均线
        technical_features['MA5'] = df['close'].rolling(5).mean()
        technical_features['MA10'] = df['close'].rolling(10).mean()
        technical_features['MA20'] = df['close'].rolling(20).mean()

        # 价格相对位置
        technical_features['价格_MA5_比'] = df['close'] / technical_features['MA5']
        technical_features['价格_MA20_比'] = df['close'] / technical_features['MA20']

        # 成交量移动平均 - 尝试不同的列名
        volume_col = None
        for col_name in ['volume', 'vol', '成交量', 'turnover']:
            if col_name in df.columns:
                volume_col = col_name
                break

        if volume_col:
            technical_features['成交量_MA5'] = df[volume_col].rolling(5).mean()
            technical_features['成交量_相对强度'] = df[volume_col] / technical_features['成交量_MA5']
        else:
            technical_features['成交量_MA5'] = 0
            technical_features['成交量_相对强度'] = 0

        # RSI (简化版)
        delta = df['close'].diff()
        gain = (delta.where(delta > 0, 0)).rolling(14).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
        rs = gain / loss
        technical_features['RSI'] = 100 - (100 / (1 + rs))

        # 布林带
        rolling_mean = df['close'].rolling(20).mean()
        rolling_std = df['close'].rolling(20).std()
        technical_features['布林上轨'] = rolling_mean + (rolling_std * 2)
        technical_features['布林下轨'] = rolling_mean - (rolling_std * 2)
        technical_features['布林位置'] = (df['close'] - technical_features['布林下轨']) / (technical_features['布林上轨'] - technical_features['布林下轨'])

        return technical_features

    def _merge_all_features(self, feature_dfs):
        """合并所有特征DataFrame"""
        # 过滤掉空的DataFrame
        valid_dfs = [df for df in feature_dfs if not df.empty]

        if not valid_dfs:
            return pd.DataFrame()

        # 找到共同的日期范围
        common_index = valid_dfs[0].index
        for df in valid_dfs[1:]:
            common_index = common_index.intersection(df.index)

        # 基于共同日期范围合并
        comprehensive_df = pd.DataFrame(index=common_index)

        for df in valid_dfs:
            comprehensive_df = comprehensive_df.join(df, how='left')

        return comprehensive_df

    def _add_time_features(self, df):
        """添加时间特征"""
        if df.empty:
            return df

        df = df.copy()

        # 时间特征
        df['年'] = df.index.year
        df['月'] = df.index.month
        df['日'] = df.index.day
        df['星期'] = df.index.dayofweek  # 0=Monday, 6=Sunday
        df['是否周末'] = (df.index.dayofweek >= 5).astype(int)
        df['季度'] = df.index.quarter
        df['年内第几天'] = df.index.dayofyear

        return df

    def _calculate_derived_features(self, df):
        """计算衍生特征"""
        if df.empty:
            return df

        df = df.copy()

        # 1. 价格动量特征
        if '收盘价' in df.columns:
            df['价格动量_3日'] = df['收盘价'].rolling(3).apply(lambda x: x.iloc[-1] - x.iloc[0] if len(x) == 3 else np.nan)
            df['价格动量_5日'] = df['收盘价'].rolling(5).apply(lambda x: x.iloc[-1] - x.iloc[0] if len(x) == 5 else np.nan)

        # 2. 情感-价格相关性特征
        if '每日情感分数' in df.columns and '日收益率' in df.columns:
            # 滚动相关性
            window = 10
            df['情感价格相关性_10日'] = df['每日情感分数'].rolling(window).corr(df['日收益率'])

            # 情感与收益率的乘积（同向性指标）
            df['情感收益同向性'] = df['每日情感分数'] * df['日收益率']

        # 3. 综合信号特征
        if '交易信号_数值' in df.columns and 'RSI' in df.columns:
            df['综合信号'] = (df['交易信号_数值'] + (df['RSI'] - 50) / 25) / 2  # 标准化RSI并结合

        # 4. 风险特征
        if '情感波动率' in df.columns and '振幅' in df.columns:
            df['综合波动率'] = (df['情感波动率'].fillna(0) + df['振幅'].fillna(0)) / 2

        return df

    def save_features(self, comprehensive_df, filename=None):
        """保存特征表格"""
        if filename is None:
            filename = f"{config.STOCK_CODE}_comprehensive_features.csv"

        filepath = os.path.join('data/processed', filename)
        comprehensive_df.to_csv(filepath, encoding='utf-8-sig')
        print(f"✅ 特征表格已保存到: {filepath}")

        return filepath

    def print_feature_summary(self, comprehensive_df):
        """打印特征总结"""
        print("\n" + "="*80)
        print("📊 综合特征表格总结")
        print("="*80)

        print(f"📅 时间范围: {comprehensive_df.index.min().strftime('%Y-%m-%d')} 至 {comprehensive_df.index.max().strftime('%Y-%m-%d')}")
        print(f"📊 数据行数: {len(comprehensive_df)} 天")
        print(f"📈 特征列数: {len(comprehensive_df.columns)} 个")

        print(f"\n🔍 特征分类统计:")

        # 按特征类型分类
        feature_categories = {
            '价格特征': ['开盘价', '最高价', '最低价', '收盘价', '成交量', '成交额', '日收益率', '价格变化额', '振幅', '成交量变化率', '量价比'],
            '情感特征': ['每日情感分数', '评论数量', '正面评论数', '负面评论数', '中性评论数', '情感强度', '情感方向', '情感变化', '情感变化率'],
            '主题特征': [col for col in comprehensive_df.columns if '_情感' in col or '主题情感' in col],
            '极端事件': [col for col in comprehensive_df.columns if '极端' in col],
            '技术指标': ['MA5', 'MA10', 'MA20', 'RSI', '布林上轨', '布林下轨', '布林位置'] + [col for col in comprehensive_df.columns if 'MA' in col or '价格_' in col],
            '动量特征': [col for col in comprehensive_df.columns if '动量' in col or '波动' in col],
            '时间特征': ['年', '月', '日', '星期', '是否周末', '季度', '年内第几天'],
            '衍生特征': [col for col in comprehensive_df.columns if '相关性' in col or '同向性' in col or '综合' in col],
            '交易信号': [col for col in comprehensive_df.columns if '信号' in col or '反转' in col or '背离' in col]
        }

        for category, features in feature_categories.items():
            existing_features = [f for f in features if f in comprehensive_df.columns]
            if existing_features:
                print(f"  {category}: {len(existing_features)} 个")
                print(f"    {', '.join(existing_features[:3])}{'...' if len(existing_features) > 3 else ''}")

        print(f"\n📋 数据质量:")
        missing_rate = comprehensive_df.isnull().sum().sum() / (len(comprehensive_df) * len(comprehensive_df.columns))
        print(f"  缺失值比例: {missing_rate:.2%}")
        print(f"  完整数据行数: {comprehensive_df.dropna().shape[0]} 行")

        print("\n" + "="*80)


def create_comprehensive_features(price_df, daily_sentiment_df, advanced_results, temporal_results):
    """
    创建综合特征表格的主函数

    返回:
    - comprehensive_df: 综合特征DataFrame
    """
    engineer = FeatureEngineer()

    # 创建综合特征
    comprehensive_df = engineer.create_comprehensive_daily_features(
        price_df, daily_sentiment_df, advanced_results, temporal_results
    )

    # 打印特征总结
    engineer.print_feature_summary(comprehensive_df)

    # 保存特征表格
    filepath = engineer.save_features(comprehensive_df)

    return comprehensive_df, filepath


In [ ]:
# 测试代码
from feature_engineering import *

# 创建综合特征表格
comprehensive_features, feature_filepath = create_comprehensive_features(
    processed_price_df,
    daily_sentiment_df,
    advanced_results,
    temporal_results
)

# 显示前几行数据
print("\n📋 综合特征表格预览:")
print(comprehensive_features.head())

# 显示特征表格的基本信息
print(f"\n📊 特征表格形状: {comprehensive_features.shape}")
print(f"📅 时间范围: {comprehensive_features.index.min()} 至 {comprehensive_features.index.max()}")

In [ ]:
feature_filepath

In [ ]:
# 运行所有预测模型
from predictive_models import run_all_models

# 首先安装必要的依赖
try:
    import torch
    print("✅ PyTorch已安装")
except ImportError:
    print("📦 正在安装PyTorch...")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "torch"])
    print("✅ PyTorch安装完成")

try:
    import statsmodels
    print("✅ Statsmodels已安装")
except ImportError:
    print("📦 正在安装Statsmodels...")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "statsmodels"])
    print("✅ Statsmodels安装完成")


In [ ]:
# 运行所有预测模型
print("🚀 开始运行预测模型...")
print(f"📊 数据形状: {comprehensive_features.shape}")
print(f"📅 时间范围: {comprehensive_features.index.min()} 至 {comprehensive_features.index.max()}")

# 运行所有模型
model_pipeline = run_all_models(
    comprehensive_features,
    target_col='日收益率',
    feature_cols=None  # 自动选择特征
)


In [ ]:
# 查看模型结果
if model_pipeline:
    print("\n🏆 模型性能总结:")
    model_pipeline.evaluator.compare_models()

    print("\n📋 可用的模型:")
    for model_name in model_pipeline.models.keys():
        print(f"  ✅ {model_name}")

    print("\n📊 预测结果:")
    for model_name, results in model_pipeline.predictions.items():
        print(f"  {model_name}: {len(results['predictions'])} 个预测点")


In [ ]:
# 详细查看最佳模型的结果
if model_pipeline and model_pipeline.predictions:
    # 获取最佳模型（按RMSE排序）
    comparison = pd.DataFrame(model_pipeline.evaluator.metrics).T.sort_values('RMSE')
    best_model = comparison.index[0]

    print(f"🥇 最佳模型: {best_model}")
    print(f"   RMSE: {comparison.loc[best_model, 'RMSE']:.6f}")
    print(f"   R²: {comparison.loc[best_model, 'R²']:.4f}")

    # 显示预测结果的前几个值
    best_results = model_pipeline.predictions[best_model]
    print(f"\n📈 {best_model} 预测结果示例:")
    print("实际值 vs 预测值:")
    for i in range(min(10, len(best_results['y_test']))):
        actual = best_results['y_test'][i]
        predicted = best_results['predictions'][i]
        print(f"  Day {i+1}: 实际={actual:.6f}, 预测={predicted:.6f}, 误差={abs(actual-predicted):.6f}")


In [ ]:
# 生成详细的模型分析报告
if model_pipeline:
    print("📄 生成详细分析报告...")

    # 保存模型结果到CSV
    results_summary = pd.DataFrame(model_pipeline.evaluator.metrics).T
    results_summary.to_csv('reports/model_performance_summary.csv', encoding='utf-8-sig')
    print("✅ 模型性能摘要已保存")

    # 生成预测结果对比表
    prediction_comparison = pd.DataFrame()
    for model_name, results in model_pipeline.predictions.items():
        pred_df = pd.DataFrame({
            f'{model_name}_actual': results['y_test'][:50],  # 只取前50个
            f'{model_name}_predicted': results['predictions'][:50]
        })
        prediction_comparison = pd.concat([prediction_comparison, pred_df], axis=1)

    prediction_comparison.to_csv('reports/prediction_comparison.csv', encoding='utf-8-sig')
    print("✅ 预测结果对比已保存")

    print("\n🎉 所有分析完成！")
    print("📁 查看reports文件夹获取详细结果")

In [ ]:
# 导入词云生成模块并运行
from wordcloud_generator import run_wordcloud_analysis, auto_find_single_word_file
import pandas as pd
import os

# 首先检查是否有包含single_word列的文件
print("🔍 搜索包含single_word列的文件...")
file_path = auto_find_single_word_file()

if file_path:
    print(f"✅ 找到文件: {file_path}")
    # 运行词云分析
    wordcloud, words_list = run_wordcloud_analysis(file_path)
else:
    print("❌ 未找到包含single_word列的文件")
    print("📋 让我检查现有的CSV文件...")

    # 检查processed目录中的文件
    processed_files = []
    if os.path.exists('data/processed/'):
        for file in os.listdir('data/processed/'):
            if file.endswith('.csv'):
                processed_files.append(file)

    print(f"📁 processed目录中的CSV文件: {processed_files}")

    # 尝试检查第一个文件的列名
    if processed_files:
        sample_file = f'data/processed/{processed_files[0]}'
        try:
            df_sample = pd.read_csv(sample_file, nrows=5)
            print(f"📊 {processed_files[0]} 的列名: {list(df_sample.columns)}")
        except Exception as e:
            print(f"❌ 读取文件失败: {e}")


In [ ]:
# 如果没有single_word列，我们可以从评论文本中创建词云
# 这里使用评论数据作为替代方案
print("🔄 尝试使用评论数据生成词云...")

try:
    # 查找评论数据文件
    comment_files = [f for f in os.listdir('data/processed/') if 'comments' in f and f.endswith('.csv')]
    print(f"📁 找到评论文件: {comment_files}")

    if comment_files:
        # 使用第一个评论文件
        comment_file = f'data/processed/{comment_files[0]}'
        print(f"📖 读取评论文件: {comment_file}")

        # 只读取前1000行避免内存问题
        df_comments = pd.read_csv(comment_file, nrows=1000)
        print(f"📊 评论数据形状: {df_comments.shape}")
        print(f"📋 评论数据列名: {list(df_comments.columns)}")

        # 查找文本相关的列
        text_columns = [col for col in df_comments.columns if any(keyword in col.lower() for keyword in ['text', 'content', 'comment', '内容'])]
        print(f"📝 找到文本列: {text_columns}")

        if text_columns:
            # 使用第一个文本列创建词云
            text_column = text_columns[0]
            print(f"✅ 使用列 '{text_column}' 生成词云")

            # 手动创建词云
            from wordcloud_generator import process_words_from_column, generate_wordcloud, generate_word_frequency_chart
            from collections import Counter
            import jieba
            import re

            # 处理文本数据
            text_data = df_comments[text_column].dropna().astype(str)
            all_words = []

            print("🔄 正在分词处理...")
            for i, text in enumerate(text_data.head(500)):  # 只处理前500条
                if i % 100 == 0:
                    print(f"   处理进度: {i}/500")

                # 使用jieba分词
                words = jieba.cut(text)
                for word in words:
                    word = word.strip()
                    if len(word) > 1 and not re.match(r'^[0-9\.\-\+\%\s\W]+$', word):
                        # 扩展的停用词列表 - 包含股票相关高频词
                        stop_words = {
                            # 基础停用词
                            '的', '了', '是', '在', '有', '和', '就', '不', '人', '都', '一', '个',
                            '上', '也', '很', '到', '说', '要', '去', '你', '会', '着', '没有', '看',
                            '好', '自己', '这', '那', '什么', '时候', '可以', '还是', '为了', '但是',
                            '因为', '如果', '虽然', '然后', '所以', '而且', '或者', '不过', '只是',
                            '已经', '可能', '应该', '比较', '非常', '特别', '一些', '这些', '那些',
                            '这样', '那样', '怎么', '为什么', '多少', '哪里', '什么时候', 'nbsp',
                            # 股票相关高频词
                            '股票', '今天', '明天', '昨天', '现在', '以后', '之前', '之后', '感觉',
                            '觉得', '认为', '估计', '应该', '肯定', '可能', '或许', '大概', '差不多',
                            '银行', '平安', '分红', '就是', '亿元', '市场', '买入', '公司', '万元',
                            '股价', '涨停', '跌停', '涨幅', '跌幅', '开盘', '收盘', '成交量', '换手率',
                            '主力', '机构', '散户', '资金', '流入', '流出', '净流入', '净流出',
                            '利好', '利空', '消息', '公告', '年报', '季报', '业绩', '盈利',
                            '投资', '投资者', '持股', '持有', '建仓', '减仓', '加仓', '清仓',
                            '看多', '看空', '看好', '看跌', '看涨', '止损', '止盈', '套牢',
                            '解套', '抄底', '逃顶', '追高', '杀跌', '反弹', '调整', '回调',
                            '突破', '支撑', '压力', '阻力', '趋势', '走势', '形态', '技术',
                            '基本面', '消息面', '政策', '央行', '降准', '降息', '加息',
                            '板块', '概念', '题材', '热点', '龙头', '妖股', '黑马', '白马',
                            '蓝筹', '小盘', '大盘', '指数', '上证', '深证', '创业板', '科创板',
                            '北向', '南向', '外资', '内资', '国资', '民营', '混改',
                            '重组', '并购', '分拆', '退市', '停牌', '复牌', '除权', '除息',
                            '配股', '增发', '回购', '质押', '解质', '减持', '增持',
                            '研报', '评级', '目标价', '估值', '市盈率', '市净率', '市销率',
                            '毛利率', '净利率', 'ROE', 'ROA', 'PB', 'PE', 'PEG',
                            '现金流', '负债率', '资产', '负债', '股本', '流通股', '总股本',
                            '股东', '股份', '控股', '参股', '子公司', '母公司', '关联',
                            '行业', '产业', '制造', '科技', '医药', '金融', '地产', '消费',
                            '新能源', '人工智能', '5G', '芯片', '半导体', '新材料',
                            '环保', '军工', '航空', '铁路', '港口', '物流', '电商',
                            '游戏', '影视', '教育', '旅游', '餐饮', '零售', '保险',
                            '券商', '信托', '基金', '私募', '公募', '理财', '债券',
                            '期货', '期权', '外汇', '黄金', '原油', '商品', '大宗',
                            '宏观', '微观', '经济', '通胀', '通缩', 'GDP', 'CPI', 'PPI',
                            '货币', '财政', '税收', '补贴', '扶持', '监管', '合规',
                            '风险', '收益', '波动', '稳定', '增长', '下滑', '复苏', '萧条'
                        }
                        if word not in stop_words:
                            all_words.append(word)

            print(f"📝 分词完成，共提取 {len(all_words)} 个词汇")
            print(f"📝 去重后共 {len(set(all_words))} 个不同词汇")

            # 显示前20个高频词
            word_freq = Counter(all_words)
            top_words = word_freq.most_common(20)
            print("🔥 前20个高频词:")
            for word, freq in top_words:
                print(f"   {word}: {freq}")

            # 生成词云图
            print("\n🎨 生成词云图...")
            if all_words:
                wordcloud = generate_wordcloud(all_words, save_path='reports/figures/comment_wordcloud.png')
                generate_word_frequency_chart(all_words, save_path='reports/figures/comment_word_frequency.png')
                print("✅ 词云图生成完成！")
            else:
                print("❌ 没有有效词汇可生成词云")
        else:
            print("❌ 未找到文本相关的列")
    else:
        print("❌ 未找到评论数据文件")

except Exception as e:
    print(f"❌ 处理过程中出错: {e}")
    import traceback
    traceback.print_exc()
